<a href="https://colab.research.google.com/github/ddopazo92/Introduccion_datascience_diego_dopazo/blob/main/Clase%2003%20TCL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''
  CREATE TABLE IF NOT EXISTS cuentas (
    id INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    saldo INTEGER NOT NULL
  )
  ''')


In [53]:
cursor = conn.cursor()

cursor.execute("SELECT name from sqlite_master WHERE type='table';")

print(cursor.fetchall())

[('cuentas',)]


In [54]:
def transferir_dinero(origen, destino, monto):
    """
    Procedimiento para transferir dinero entre cuentas.
    Utiliza TCL: BEGIN TRANSACTION, COMMIT y ROLLBACK.
    """
    try:
        # Iniciar la transacción
        cursor.execute("BEGIN TRANSACTION;")

        # Verificar saldo suficiente
        cursor.execute("SELECT saldo FROM cuentas WHERE id = ?", (origen,))
        saldo_origen = cursor.fetchone()[0]

        if saldo_origen < monto:
            raise Exception("Saldo insuficiente")

        # Restar saldo de la cuenta de origen
        cursor.execute("UPDATE cuentas SET saldo = saldo - ? WHERE id = ?", (monto, origen))

        # Sumar saldo a la cuenta de destino
        cursor.execute("UPDATE cuentas SET saldo = saldo + ? WHERE id = ?", (monto, destino))

        # Confirmar la transacción
        conn.commit()
        print(f"Transferencia de ${monto} completada con éxito.")

    except Exception as e:
        # Revertir la transacción en caso de error
        conn.rollback()
        print(f"Error en la transferencia: {e}")

In [55]:
# Insertar cuentas de prueba
cursor.executemany('''
    INSERT INTO cuentas (id, nombre, saldo) VALUES (?, ?, ?)
''', [(1, 'Cuenta A', 1000.00), (2, 'Cuenta B', 500.00)])

conn.commit()

In [56]:
# Consultar los saldos antes de la transacción
cursor.execute("SELECT * FROM cuentas")
print(cursor.fetchall())

[(1, 'Cuenta A', 1000), (2, 'Cuenta B', 500)]


In [57]:
# Prueba la transferencia
transferir_dinero(1, 2, 300)

Transferencia de $300 completada con éxito.


In [58]:
# Consultar los saldos antes de la transacción
cursor.execute("SELECT * FROM cuentas")
print(cursor.fetchall())

[(1, 'Cuenta A', 700), (2, 'Cuenta B', 800)]


In [59]:
# Prueba la transferencia
transferir_dinero(1, 2, 700)

Transferencia de $700 completada con éxito.


In [60]:
# Consultar los saldos antes de la transacción
cursor.execute("SELECT * FROM cuentas")
print(cursor.fetchall())

[(1, 'Cuenta A', 0), (2, 'Cuenta B', 1500)]


In [61]:
# Prueba la transferencia
transferir_dinero(1, 2, 500)

Error en la transferencia: Saldo insuficiente


In [62]:
# Cerrar conexión
conn.close()